In [2]:

import numpy as np
import pandas as pd
import pickle
import faiss
from scipy.sparse import load_npz
import time

print("=" * 60)
print("HYBRID FUSION - PRODUCTION GRADE")
print("=" * 60)


print("\n" + "-" * 40)
print("Loading CBF models...")
print("-" * 40)


# Load CBF artifacts
with open ("../models/cbf/tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

with open ("../models/cbf/svd_model.pkl", "rb") as f:
    svd = pickle.load(f)

index = faiss.read_index("../models/cbf/faiss_index.bin")

with open ("../models/cbf/movie_id_to_idx.pkl", "rb") as f:
    movie_id_to_idx = pickle.load(f)

with open('../models/cbf/idx_to_movie_id.pkl', 'rb') as f:
    idx_to_movie_id = pickle.load(f)

with open('../models/cbf/genre_vectors.pkl', 'rb') as f:
    genre_vectors = pickle.load(f)

# Load movies dataframe
movies_df = pd.read_pickle('../models/cbf/movies_df.pkl')

print(f"✅ CBF models loaded:")
print(f"   - Movies in index: {index.ntotal}")
print(f"   - SVD dimension: {svd.components_.shape[0]}")
print(f"   - Genre vectors: {len(genre_vectors)}")

HYBRID FUSION - PRODUCTION GRADE

----------------------------------------
Loading CBF models...
----------------------------------------
✅ CBF models loaded:
   - Movies in index: 45423
   - SVD dimension: 200
   - Genre vectors: 20


In [3]:
print("\n" + "-" * 40)
print("Loading CF models...")
print("-" * 40)

# Load CF artifacts
with open('../models/cf/als_model.pkl', 'rb') as f:
    als_model = pickle.load(f)

with open('../models/cf/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('../models/cf/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

with open('../models/cf/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)

# Load confidence matrix for user history
C = load_npz('../data/processed/als_confidence_matrix.npz')

print(f"✅ CF models loaded:")
print(f"   - Users in mapper: {len(user_mapper)}")
print(f"   - Movies in mapper: {len(movie_mapper)}")
print(f"   - ALS factors: {als_model.user_factors.shape[1]}")


----------------------------------------
Loading CF models...
----------------------------------------


c:\Users\ADEGOKE\anaconda3\envs\Recommender\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ CF models loaded:
   - Users in mapper: 270882
   - Movies in mapper: 45423
   - ALS factors: 100


In [4]:
print("\n" + "-" * 40)
print("Pre-computing global CF statistics (memory-safe)...")
print("-" * 40)

sample_size = min(2000, len(user_mapper)) 
sample_users = np.random.choice(list(user_mapper.values()), sample_size, replace=False)

# Parameters for memory efficiency
n_movies = als_model.item_factors.shape[0]
scores_per_user = 500  # Only sample 500 movies per user instead of all 45k

print(f"   Sampling {sample_size} users × {scores_per_user} movies = {sample_size * scores_per_user:,} scores")

# Pre-allocate numpy array (much more memory efficient than Python list)
all_scores = np.zeros(sample_size * scores_per_user, dtype=np.float32)

idx = 0
for user_idx in sample_users:
    user_vector = als_model.user_factors[user_idx]
    
    # Randomly sample movie indices instead of using all
    movie_indices = np.random.choice(n_movies, size=scores_per_user, replace=False)
    sampled_scores = np.dot(als_model.item_factors[movie_indices], user_vector)
    
    # Store in pre-allocated array
    all_scores[idx:idx + scores_per_user] = sampled_scores
    idx += scores_per_user
    
    # Progress indicator
    if (idx // scores_per_user) % 200 == 0:
        print(f"   Processed {idx // scores_per_user}/{sample_size} users")

# Calculate percentiles on the sampled array
global_cf_min = float(np.percentile(all_scores, 1))
global_cf_max = float(np.percentile(all_scores, 99))
global_cf_mean = float(np.mean(all_scores))
global_cf_std = float(np.std(all_scores))

# Clean up to free memory
del all_scores

print(f"\n✅ Global CF statistics (from {sample_size * scores_per_user:,} samples):")
print(f"   - Min (1%): {global_cf_min:.4f}")
print(f"   - Max (99%): {global_cf_max:.4f}")
print(f"   - Mean: {global_cf_mean:.4f}")
print(f"   - Std: {global_cf_std:.4f}")


----------------------------------------
Pre-computing global CF statistics (memory-safe)...
----------------------------------------
   Sampling 2000 users × 500 movies = 1,000,000 scores
   Processed 200/2000 users
   Processed 400/2000 users
   Processed 600/2000 users
   Processed 800/2000 users
   Processed 1000/2000 users
   Processed 1200/2000 users
   Processed 1400/2000 users
   Processed 1600/2000 users
   Processed 1800/2000 users
   Processed 2000/2000 users

✅ Global CF statistics (from 1,000,000 samples):
   - Min (1%): -0.0883
   - Max (99%): 0.3996
   - Mean: 0.0106
   - Std: 0.0809


In [7]:
print("\n" + "-" * 40)
print("Pre-computing movie popularity...")
print("-" * 40)

movie_popularity = {}
for movie_id, idx in movie_mapper.items():
    popularity = C[idx].nnz  # Number of users who interacted
    movie_popularity[movie_id] = popularity

max_pop = max(movie_popularity.values()) if movie_popularity else 1
movie_popularity_norm = {mid: pop/max_pop for mid, pop in movie_popularity.items()}
print(f"Movie popularity computed for {len(movie_popularity_norm)} movies")



----------------------------------------
Pre-computing movie popularity...
----------------------------------------
Movie popularity computed for 45423 movies


In [8]:
def get_cbf_candidates(movie_title, n_candidates=200):  
    """
    Stage 1: Get top-N most similar movies from CBF
    Returns: list of (movie_id, similarity_score, rank)
    """
    # Find movie in dataframe
    movie_row = movies_df[movies_df['title'].str.lower() == movie_title.lower()]
    
    if len(movie_row) == 0:
        print(f"⚠ Movie not found: {movie_title}")
        return []
    
    movie_id = movie_row.iloc[0]['id']
    
    if movie_id not in movie_id_to_idx:
        print(f"⚠ Movie ID {movie_id} not in index")
        return []
    
    # Get movie vector
    movie_idx = movie_id_to_idx[movie_id]
    
    # Load movie's combined content and transform
    movie_content = movie_row.iloc[0]['combined_content']
    tfidf_vec = tfidf.transform([movie_content])
    svd_vec = svd.transform(tfidf_vec).astype('float32')
    
    # Ensure float32 and normalized
    svd_vec = np.ascontiguousarray(svd_vec, dtype=np.float32)
    faiss.normalize_L2(svd_vec)
    
    # Search FAISS index
    distances, indices = index.search(svd_vec, n_candidates + 1)
    
    # Convert to results (skip first which is the movie itself)
    candidates = []
    for i, idx in enumerate(indices[0][1:]):
        candidate_id = idx_to_movie_id[idx]
        candidates.append({
            'movie_id': candidate_id,
            'cbf_similarity': float(distances[0][i+1]),
            'cbf_rank': i+1
        })
    
    return candidates

def get_cf_scores(user_id, candidate_movies):
    """
    Stage 2: Score candidates with CF
    Returns: dict of {movie_id: cf_score}
    """
    if user_id not in user_mapper:
        # Cold user - return popularity-based scores
        return {movie_id: movie_popularity_norm.get(movie_id, 0.0) 
                for movie_id in candidate_movies}
    
    user_idx = user_mapper[user_id]
    user_vector = als_model.user_factors[user_idx]
    
    scores = {}
    for movie_id in candidate_movies:
        if movie_id in movie_mapper:
            movie_idx = movie_mapper[movie_id]
            score = np.dot(als_model.item_factors[movie_idx], user_vector)
            scores[movie_id] = float(score)
        else:
            # Unknown movie - use popularity
            scores[movie_id] = movie_popularity_norm.get(movie_id, 0.0)
    
    return scores

def normalize_scores_global(scores_dict):
    """
    FIX: Global normalization using pre-computed statistics
    Normalizes to [0, 1] range based on global CF distribution
    """
    if not scores_dict:
        return {}
    
    normalized = {}
    for k, v in scores_dict.items():
        # Clip to global range then normalize
        clipped = np.clip(v, global_cf_min, global_cf_max)
        normalized[k] = (clipped - global_cf_min) / (global_cf_max - global_cf_min)
    
    return normalized

def get_alpha_smoothed(user_id):
    """
    FIX: Smooth alpha using sigmoid function instead of discrete steps
    """
    if user_id not in user_mapper:
        return 0.0
    
    user_idx = user_mapper[user_id]
    n_interactions = C[:, user_idx].nnz
    
    # Smooth sigmoid function: alpha = 0.8 / (1 + e^(-0.2*(n-15)))
    # This gives smooth transition from 0 to 0.8
    alpha = 0.8 / (1 + np.exp(-0.2 * (n_interactions - 15)))
    
    return float(alpha)


In [9]:
print("\n" + "-" * 40)
print("Testing Hybrid Fusion Pipeline (with all fixes)")
print("-" * 40)

# Test parameters
test_movie = "toy story"
test_user = 38150  # User with history
n_candidates = 200  # Increased candidate pool
n_final = 12

print(f"\nQuery: '{test_movie}'")
print(f"User ID: {test_user}")

# STAGE 1: CBF retrieves candidates (200)
print("\n🔍 Stage 1: CBF Candidate Retrieval")
start = time.time()
candidates = get_cbf_candidates(test_movie, n_candidates)
cbf_time = time.time() - start

print(f"   Found {len(candidates)} candidates in {cbf_time*1000:.2f} ms")
print(f"   Top 5 CBF candidates:")
for i, cand in enumerate(candidates[:5]):
    title = movies_df[movies_df['id'] == cand['movie_id']]['title'].values[0]
    print(f"     {i+1}. {title} (sim: {cand['cbf_similarity']:.4f})")

# STAGE 2: CF scores candidates
print("\n🔢 Stage 2: CF Scoring")
start = time.time()
candidate_ids = [c['movie_id'] for c in candidates]
cf_scores_raw = get_cf_scores(test_user, candidate_ids)
cf_time = time.time() - start

print(f"   Scored {len(cf_scores_raw)} candidates in {cf_time*1000:.2f} ms")

# Use global normalization
cf_scores_norm = normalize_scores_global(cf_scores_raw)

# CBF rank scores (still normalized per query - this is fine)
cbf_rank_scores = {c['movie_id']: (n_candidates + 1 - c['cbf_rank']) / n_candidates 
                   for c in candidates}

# Use smoothed alpha
alpha = get_alpha_smoothed(test_user)
print(f"\n⚖ Smoothed alpha: {alpha:.3f}")

# Hybrid fusion
hybrid_scores = {}
for movie_id in candidate_ids:
    cbf_score = cbf_rank_scores.get(movie_id, 0)
    cf_score = cf_scores_norm.get(movie_id, 0)
    hybrid_scores[movie_id] = alpha * cf_score + (1 - alpha) * cbf_score

# Sort and get top N
final_recommendations = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n_final]

print(f"\n🎯 Final Top {n_final} Hybrid Recommendations:")
for i, (movie_id, score) in enumerate(final_recommendations):
    title = movies_df[movies_df['id'] == movie_id]['title'].values[0]
    cbf_rank = next((c['cbf_rank'] for c in candidates if c['movie_id'] == movie_id), None)
    cf_raw = cf_scores_raw.get(movie_id, 0)
    print(f"   {i+1:2d}. {title[:40]:40} | Hybrid: {score:.3f} | CBF Rank: {cbf_rank:3d} | CF: {cf_raw:.3f}")



----------------------------------------
Testing Hybrid Fusion Pipeline (with all fixes)
----------------------------------------

Query: 'toy story'
User ID: 38150

🔍 Stage 1: CBF Candidate Retrieval
   Found 200 candidates in 1729.65 ms
   Top 5 CBF candidates:
     1. Toy Story 2 (sim: 0.9344)
     2. Toy Story of Terror! (sim: 0.8369)
     3. Toy Story 3 (sim: 0.7051)
     4. Free Birds (sim: 0.6879)
     5. Partysaurus Rex (sim: 0.6762)

🔢 Stage 2: CF Scoring
   Scored 200 candidates in 3.77 ms

⚖ Smoothed alpha: 0.800

🎯 Final Top 12 Hybrid Recommendations:
    1. The Wrong Trousers                       | Hybrid: 0.893 | CBF Rank: 108 | CF: 0.937
    2. A Close Shave                            | Hybrid: 0.876 | CBF Rank: 125 | CF: 0.948
    3. The Santa Clause                         | Hybrid: 0.873 | CBF Rank:  89 | CF: 0.376
    4. Jungle 2 Jungle                          | Hybrid: 0.732 | CBF Rank:  44 | CF: 0.263
    5. Pete's Dragon                            | Hybrid: 0.6

In [10]:
print("\n" + "-" * 40)
print("Cold Start Test (New User - With Popularity Fallback)")
print("-" * 40)

cold_user = 999999999  # Non-existent user
cold_alpha = get_alpha_smoothed(cold_user)
print(f"Smoothed alpha for new user: {cold_alpha:.3f}")

cold_cf_scores = get_cf_scores(cold_user, candidate_ids)
# Popularity scores, not zeros!

cold_hybrid_scores = {}
for movie_id in candidate_ids:
    cbf_score = cbf_rank_scores.get(movie_id, 0)
    cf_score = cold_cf_scores.get(movie_id, 0)
    cold_hybrid_scores[movie_id] = cold_alpha * cf_score + (1 - cold_alpha) * cbf_score

cold_final = sorted(cold_hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n_final]

print(f"\nTop {n_final} for New User (CBF + Popularity):")
for i, (movie_id, score) in enumerate(cold_final):
    title = movies_df[movies_df['id'] == movie_id]['title'].values[0]
    pop_score = cold_cf_scores.get(movie_id, 0)
    print(f"   {i+1:2d}. {title[:40]:40} | Hybrid: {score:.3f} | Popularity: {pop_score:.3f}")


----------------------------------------
Cold Start Test (New User - With Popularity Fallback)
----------------------------------------
Smoothed alpha for new user: 0.000

Top 12 for New User (CBF + Popularity):
    1. Toy Story 2                              | Hybrid: 1.000 | Popularity: 0.309
    2. Toy Story of Terror!                     | Hybrid: 0.995 | Popularity: 0.002
    3. Toy Story 3                              | Hybrid: 0.990 | Popularity: 0.140
    4. Free Birds                               | Hybrid: 0.985 | Popularity: 0.001
    5. Partysaurus Rex                          | Hybrid: 0.980 | Popularity: 0.000
    6. The Happy Elf                            | Hybrid: 0.975 | Popularity: 0.000
    7. The Boss Baby                            | Hybrid: 0.970 | Popularity: 0.002
    8. Superstar Goofy                          | Hybrid: 0.965 | Popularity: 0.000
    9. The Madagascar Penguins in a Christmas C | Hybrid: 0.960 | Popularity: 0.001
   10. Kronk's New Groove      

In [11]:
print("\n" + "-" * 40)
print("Performance Monitoring Metrics")
print("-" * 40)

# Simplified precision@k calculation for this query
# In production, we run this on a holdout set
relevant_movies = [862, 3114, 2054]  # Example: Toy Story, Toy Story 2, etc.

def precision_at_k(recommended_ids, relevant_ids, k):
    if len(recommended_ids) > k:
        recommended_ids = recommended_ids[:k]
    hits = len(set(recommended_ids) & set(relevant_ids))
    return hits / k

hybrid_ids = [m[0] for m in final_recommendations]
cbf_ids = [c['movie_id'] for c in candidates[:n_final]]

print(f"Precision@10 for this query:")
print(f"   - Pure CBF: {precision_at_k(cbf_ids, relevant_movies, 10):.3f}")
print(f"   - Hybrid:   {precision_at_k(hybrid_ids, relevant_movies, 10):.3f}")

print("\n" + "=" * 60)
print("SECTION 16 COMPLETE - PRODUCTION-GRADE HYBRID FUSION")
print("=" * 60)



----------------------------------------
Performance Monitoring Metrics
----------------------------------------
Precision@10 for this query:
   - Pure CBF: 0.000
   - Hybrid:   0.000

SECTION 16 COMPLETE - PRODUCTION-GRADE HYBRID FUSION


In [13]:
import datetime
import pickle
import os
import numpy as np

print("\n" + "-" * 40)
print("💾 SAVING HYBRID FUSION ARTIFACTS")
print("-" * 40)

# Verify all required variables exist
required_vars = {
    'global_cf_min': global_cf_min,
    'global_cf_max': global_cf_max,
    'global_cf_mean': global_cf_mean,
    'global_cf_std': global_cf_std,
    'movie_popularity_norm': movie_popularity_norm
}

for name, var in required_vars.items():
    if var is None:
        raise ValueError(f"❌ {name} is None - cannot save hybrid config")
    print(f"✅ {name}: {type(var).__name__}")

# Create hybrid configuration bundle
hybrid_bundle = {
    'metadata': {
        'timestamp': datetime.datetime.now().isoformat(),
        'version': '1.1.0-production',
        'faiss_normalized': True,  # FIX: Add contiguity flag
        'faiss_contiguous': True,
        'cbf_svd_dim': 200,
        'cf_factors': 100
    },
    'stats': {
        'global_cf_min': float(global_cf_min),
        'global_cf_max': float(global_cf_max),
        'global_cf_mean': float(global_cf_mean),  # FIX: Added
        'global_cf_std': float(global_cf_std),    # FIX: Added
        'popularity_max': float(max(movie_popularity_norm.values()))  # FIX: Added
    },
    'lookups': {
        'movie_popularity_norm': movie_popularity_norm,
    },
    'hyperparams': {
        'candidate_pool_size': 200,
        'alpha_params': {
            'max': 0.8,
            'inflection': 15,
            'smoothing': 0.2
        }
    }
}

# Create models/hybrid directory if it doesn't exist
os.makedirs('../models/hybrid', exist_ok=True)

# Save to file
save_path = '../models/hybrid/hybrid_config.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(hybrid_bundle, f)

print(f"\n✅ Hybrid config saved to: {save_path}")
print(f"   Size: {os.path.getsize(save_path) / 1024:.2f} KB")

# Quick verification
print("\n🔍 Verification:")
print(f"   - global_cf_min: {hybrid_bundle['stats']['global_cf_min']:.4f}")
print(f"   - global_cf_max: {hybrid_bundle['stats']['global_cf_max']:.4f}")
print(f"   - global_cf_mean: {hybrid_bundle['stats']['global_cf_mean']:.4f}")
print(f"   - global_cf_std: {hybrid_bundle['stats']['global_cf_std']:.4f}")
print(f"   - Movies with popularity: {len(hybrid_bundle['lookups']['movie_popularity_norm'])}")



----------------------------------------
💾 SAVING HYBRID FUSION ARTIFACTS
----------------------------------------
✅ global_cf_min: float
✅ global_cf_max: float
✅ global_cf_mean: float
✅ global_cf_std: float
✅ movie_popularity_norm: dict

✅ Hybrid config saved to: ../models/hybrid/hybrid_config.pkl
   Size: 574.69 KB

🔍 Verification:
   - global_cf_min: -0.0883
   - global_cf_max: 0.3996
   - global_cf_mean: 0.0106
   - global_cf_std: 0.0809
   - Movies with popularity: 45423


In [14]:
# Quick test of CBF speed with caching
print("\nTesting CBF with repeated query:")
for i in range(5):
    start = time.time()
    _ = get_cbf_candidates("toy story", 50)
    print(f"  Run {i+1}: {(time.time()-start)*1000:.2f} ms")


Testing CBF with repeated query:
  Run 1: 2492.69 ms
  Run 2: 120.65 ms
  Run 3: 127.37 ms
  Run 4: 131.85 ms
  Run 5: 102.01 ms
